### Changes made for after_generation

* Updated the Constants; `gen-type = 'after' and version = 'v1'`
* Updated prompt to **refactor** code to embed watermark
* Updated the param of **prompt** to **code** in prompt template

In [ ]:
# !pip install google-genai -q
# !pip install scipy

In [ ]:
import pandas as pd
import numpy as np

## Constants

In [ ]:
MODEL_NAME = "gemini"
MODEL_PATH = "gemini-2.5-flash"

# Project root directory (absolute path to avoid confusion with relative paths)
BASE_DIR = '/home/fahad/Documents/MASTERS/CODEBLOCKS/Masters-Thesis-Code/promptMark' # or '.'

# Derived paths
DATASET_FILE = f'sanitized-mbpp-sample-100.json'
DATASET_PATH = f'{BASE_DIR}/datasets/core/{DATASET_FILE}'

Z_THRESHOLD = 2.120 # 97.5% confidence
GAMMA = 0.396158  # From NLTK Brown corpus
SEED_KEY = "exp2025"
SMALL_SAMPLE_THRESHOLD = 10
P_THRESHOLD = 0.05

In [ ]:
df = pd.read_json(DATASET_PATH, lines=True)
df = df[20:40]

In [ ]:
# Experiment parameters
EXPERIMENT_NUMBER = 'exp1-1'
EXP_VERSION = 'v1'
GENERATION_TYPE = 'after'  # 'during' or 'after'
SAMPLE_SIZE = df.shape[0]
DATASET = 'mbpp'

RESULTS_CSV = f'{BASE_DIR}/results/raw/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}.csv'
OUTPUT_DIR = f'{BASE_DIR}/output/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}'

In [ ]:
import hashlib
import random

# English alphabets
alphabet = list('abcdefghijklmnopqrstuvwxyz')

# Use EXPERIMENT_NUMBER as seed
seed_value = int(hashlib.md5(SEED_KEY.encode()).hexdigest(), 16)
print("SEED VALUE: ", seed_value)
random.seed(seed_value)

# Shuffle the alphabet randomly
random.shuffle(alphabet)

# Divide into two equal halves
half1 = set(alphabet[:13])
half2 = set(alphabet[13:])

# Use seed_hash to decide green and red halves
seed_hash = seed_value % 2

if seed_hash == 0:
    green_letters = half1
    red_letters = half2
else:
    green_letters = half2
    red_letters = half1

print("GREEN LETTERS:", green_letters)
print("RED LETTERS:", red_letters)

## Helper Methods

In [ ]:
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins

BUILTIN_NAMES = set(dir(builtins))

class APIVisitor(ast.NodeVisitor):
    def __init__(self):
        # Separate buckets
        self.public_classes = set()
        self.non_public_classes = set()
        self.public_funcs = set()
        self.non_public_funcs = set()
        self.public_vars = set()
        self.non_public_vars = set()

    def visit_FunctionDef(self, node):
        name = node.name

        # Dunder methods go to non-public
        if name.startswith("__") and name.endswith("__"):
            self.public_classes.add(name)
        elif name.startswith("_"):
            self.non_public_funcs.add(name)
        else:
            self.public_funcs.add(name)

        # Parameters
        for arg in node.args.args:
            if arg.arg in {"self", "cls"}:
                self.public_vars.add(arg.arg)   # treat self/cls as public
            else:
                self.non_public_vars.add(arg.arg)

        self.generic_visit(node)

    def visit_Call(self, node):
        # Skip calls to built-in functions
        if isinstance(node.func, ast.Name) and node.func.id in BUILTIN_NAMES:
            return
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        if node.name.startswith("_"):
            self.non_public_classes.add(node.name)
        else:
            self.public_classes.add(node.name)
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name):
                if target.id.isupper():
                    self.public_vars.add(target.id)  # constants
                else:
                    self.non_public_vars.add(target.id)
        self.generic_visit(node)

    def visit_Attribute(self, node):
        # Only consider instance attributes (self.something)
        if isinstance(node.value, ast.Name) and node.value.id in {"self", "cls"}:
            if node.attr not in BUILTIN_NAMES:  # ignore Python built-ins
                if node.attr.startswith("_"):
                    self.non_public_vars.add(node.attr)
                else:
                    self.public_vars.add(node.attr)  # treat instance public by default
        self.generic_visit(node)

def load_generated_code(file_path):
    if not os.path.exists(file_path):
        return None
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    return content.strip() if content.strip() else None

def run_code_tests(code, test_cases):
    results = []
    temp_file = 'temp_test.py'
    
    with open(temp_file, 'w', encoding='utf-8') as f:
        f.write(code + '\n')
    
    try:
        spec = importlib.util.spec_from_file_location("temp_test", temp_file)
        module = importlib.util.module_from_spec(spec)
        sys.modules["temp_test"] = module
        spec.loader.exec_module(module)
        
        globals().update(vars(module))
        
        for idx, test in enumerate(test_cases, 1):
            try:
                exec(test, globals())
                results.append((idx, True, None))
            except Exception as e:
                results.append((idx, False, str(e)))
    
    except Exception as e:
        for idx in range(len(test_cases)):
            results.append((idx+1, False, f"Code error: {str(e)}"))
    
    finally:
        if os.path.exists(temp_file):
            os.remove(temp_file)
    
    return results

In [ ]:
import math
from scipy.stats import binomtest  # Add this import

# Assuming these constants are defined elsewhere; add if needed
# GAMMA = 0.5
# Z_THRESHOLD = 2.03  # Your example threshold
# P_THRESHOLD = 0.05  # New: p-value threshold for small samples
# SMALL_SAMPLE_THRESHOLD = 10  # New: below this, use p-value
# green_letters = set(['a', 'b', ...])  # Your green letters
# red_letters = set(['x', 'y', ...])  # Your red letters

def calculate_z_score(token_count, green_count, gamma=GAMMA):
    if token_count == 0 or gamma <= 0 or gamma >= 1:
        return float("nan")
    return (green_count - gamma * token_count) / math.sqrt(gamma * (1 - gamma) * token_count)

def calculate_p_value(green_count, token_count, gamma=GAMMA):
    # Exact binomial p-value for testing if green_count is greater than expected
    if token_count == 0:
        return float("nan")
    return binomtest(green_count, token_count, gamma, alternative='greater').pvalue

def detect_watermark(original_code, generated_code):
    """
    Detect watermark based on preferred starting letters for identifiers.
    Compares original and generated code to measure deviation towards watermarking rules.
    Now uses p-value for small samples to improve detection accuracy.
    """
    import ast
    
    def get_starting_letters(code):
        try:
            tree = ast.parse(code)
        except SyntaxError:
            return set(), float("nan"), float("nan")  # Return z and p
        
        visitor = APIVisitor()
        visitor.visit(tree)
        
        # Collect non-public identifiers (user-defined variables, functions, etc.)
        non_public_tokens = (
            visitor.non_public_classes | 
            visitor.non_public_funcs | 
            visitor.non_public_vars
        )
        
        # Get starting letters, lowercase, excluding common ones like 'self', 'cls'
        relevant_tokens = {token for token in non_public_tokens if token not in {'self', 'cls'}}
        starting_letters = [word[0].lower() for word in relevant_tokens if word and word[0].isalpha()]

        green_count = sum(1 for letter in starting_letters if letter in green_letters)
        token_count = len(starting_letters)

        print("ALL STARTING LETTERS:", token_count)
        print("GREEN COUNT:", green_count)

        z_stat = calculate_z_score(token_count, green_count)
        p_stat = calculate_p_value(green_count, token_count)

        return set(starting_letters), green_count, z_stat, p_stat

    orig_starts, orig_green_count, orig_z, orig_p = get_starting_letters(original_code)
    gen_starts, gen_green_count, gen_z, gen_p = get_starting_letters(generated_code)

    preferred = green_letters
    avoided = red_letters
    
    # Calculate ratios
    orig_preferred_ratio = len(orig_starts & preferred) / len(orig_starts) if orig_starts else 0
    gen_preferred_ratio = len(gen_starts & preferred) / len(gen_starts) if gen_starts else 0
    
    orig_avoided_count = len(orig_starts & avoided)
    gen_avoided_count = len(gen_starts & avoided)

    # Adaptive detection: Use p-value for small samples, z-score for large
    def is_watermarked(z, p, token_count, green_count):
        green_fraction = (token_count - (token_count - sum(1 for _ in range(token_count) if _ < green_count))) / token_count if token_count > 0 else 0  # Approximate
        if green_fraction >= 1.0:  # Fallback: Perfect green fraction always watermarked
            return True

        if token_count < SMALL_SAMPLE_THRESHOLD:
            return p < P_THRESHOLD  # Use p-value for small samples
        else:
            return z >= Z_THRESHOLD  # Use z-score for larger samples
    
    return {
        "original_preferred_ratio": orig_preferred_ratio,
        "generated_preferred_ratio": gen_preferred_ratio,
        "original_z_score": orig_z,
        "generated_z_score": gen_z,
        "original_p_value": orig_p,
        "generated_p_value": gen_p,
        "original_is_watermarked": is_watermarked(orig_z, orig_p, len(orig_starts), orig_green_count),
        "generated_is_watermarked": is_watermarked(gen_z, gen_p, len(gen_starts), gen_green_count),
        "original_avoided_count": orig_avoided_count,
        "generated_avoided_count": gen_avoided_count,
        "original_unique_starts": sorted(orig_starts),
        "generated_unique_starts": sorted(gen_starts)
    }

## Watermark Embedding

### Gemini-2.5-Flash

In [ ]:
from google import genai
from google.genai import types
import os
import re
from pathlib import Path
import ast

API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("API_KEY")

if not API_KEY:
    raise RuntimeError(
        "Missing API key. Please set GEMINI_API_KEY or API_KEY in your environment (e.g., .env)."
    )

client = genai.Client(api_key=API_KEY)

## PROMPT TEMPLATE REFERENCE:

```tex
Wang, C. Y., DaghighFarsoodeh, A., & Pham, H. V. (2024). 
Selection of prompt engineering techniques for code generation through predicting code complexity. 
arXiv preprint arXiv:2409.16416.```

In [ ]:
SYSTEM_PROMPT = '''
### Green Letter List: {green_words}
### Red Letter List: {red_words}
### Command:
Refactor the provided code following these instructions:
    1. Green Letter Enriched Identifier: Rename identifiers (local variables, function parameters, private helper functions, internal class attributes, and temporary variables) to prefer those starting with letters from the 'Green Letter List'. Use them naturally and consistently, avoiding Red List letters where possible.
    2. Correct & Relevant: Do not change the functionality of the code. Ensure the refactored code is correct and passes the given test cases.
    3. Avoiding Instruction: Do not add docstrings. Add brief comments only to clarify complex logic, but do not over-comment.
    4. Important: Keep the method names as per the test cases, but refactor internal identifiers.
    5. Warning: Do not mention or explain the renaming rules in your output.
'''

# Prompt templates
PROBLEM_TEMPLATE = (
    "You are a helpful code assistant that can teach a junior developer how to code. Your language of choice is Python. Refactor the following Python code by embedding a watermark through identifier renaming, ensuring it remains functional and passes the test cases. Only output the refactored Python code enclosed in ```python```.\n\n"
    "##Original Code:\n{code}\n\n"
    "##Test Cases:\n{tests}\n\n"
)

def generate_code(record):
    code = record["code"]
    tests = "\n".join(record["test_list"]) 
    system_instruction = SYSTEM_PROMPT.format(green_words=green_letters, red_words=red_letters)
    full_prompt = PROBLEM_TEMPLATE.format(code=code, tests=tests)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=full_prompt)],
        )
    ]

    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.4,    # Balanced creativity vs consistency
    )

    response = client.models.generate_content(
        model=MODEL_PATH,
        contents=contents,
        config=config
    )

    text = response.text
    
    # Find all python code blocks
    code_blocks = re.findall(r'```python(.*?)```', text, re.DOTALL)

    # Get the last (actual code) block, or the one with actual content
    actual_code_blocks = [block for block in code_blocks if block.strip()]
    if actual_code_blocks:
        text = actual_code_blocks[-1].strip()  # Get the last non-empty block
    else:
        print("No actual code blocks found")
        text = ''

    print('Generated:', len(text))
    print('=' * 50)
    return (text or "").strip()

### Execution

In [ ]:
import time
def process_dataset(df, output_dir):
    Path(output_dir).parent.mkdir(exist_ok=True)
    results = []
    
    for idx, record in df.iterrows():
        task_id = record.get('task_id') or record.get('id') 
        output_file = Path(output_dir) / f"{task_id}.py"
        original_code = record.get('code', '')

        try:
            time.sleep(1)
            generated_code = generate_code(record)
            output_file.parent.mkdir(parents=True, exist_ok=True)  # Ensure directory exists
            
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(generated_code)
        except Exception as e:
            print(f"Generation failed for {task_id}: {e}")
            if not output_file.exists():
                results.append({
                    'task_id': task_id, 
                    'status': 'failed',
                    'tests_passed': 0, 'tests_failed': 0, 'total_tests': 0,
                    'pass_rate': 0, 'all_passed': False
                })
                continue

        if output_file.exists():
            generated_code = output_file.read_text(encoding='utf-8')
            
            watermark_analysis = detect_watermark(original_code, generated_code)
            
            test_results = run_code_tests(generated_code, record['test_list'])
            passed = sum(1 for _, success, _ in test_results if success)
            total = len(test_results)
            failed = total - passed
            pass_rate = (passed / total * 100) if total > 0 else 0
            all_passed = (passed == total)
            
            result = {
                'task_id': task_id,
                'tests_passed': passed, 'tests_failed': failed, 'total_tests': total,
                'pass_rate': round(pass_rate, 2), 'all_passed': all_passed,
                **watermark_analysis
            }
            results.append(result)
        else:
            results.append({
                'task_id': task_id, 'status': 'missing',
                'tests_passed': 0, 'tests_failed': 0, 'total_tests': 0,
                'pass_rate': 0, 'all_passed': False
            })

        print(f"{idx+1}/{len(df)} processed: {task_id}")
        
    results_df = pd.DataFrame(results)
    Path(RESULTS_CSV).parent.mkdir(parents=True, exist_ok=True)    
    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"Results saved to {RESULTS_CSV}")
    return results

In [ ]:
results = process_dataset(df, OUTPUT_DIR)

## Watermark Detection

- nltk:brown: total_words=1,013,640  gamma=**0.396158**
- nltk:gutenberg: total_words=2,136,080  gamma=**0.399109**
- nltk:webtext: total_words=306,832  gamma=**0.323187**

### Calculate Final Results

In [ ]:
analysis_results = []

for _, row in df.iterrows():
    task_id = row['task_id']
    original_code = row['code']
    
    print(f'Task: {task_id}')
    
    generated_file = Path(OUTPUT_DIR) / f"{task_id}.py"
    if not generated_file.exists():
        print(f"Missing file for {task_id}")
        continue

    generated_code = generated_file.read_text(encoding='utf-8')
    watermark_result = detect_watermark(original_code, generated_code)
    
    test_results = run_code_tests(generated_code, row['test_list'])
    passed = sum(1 for _, success, _ in test_results if success)
    total = len(test_results)
    
    result = {
        'task_id': task_id,
        'tests_passed': passed, 'tests_failed': total - passed, 'total_tests': total,
        'pass_rate': round((passed / total * 100) if total > 0 else 0, 2),
        'all_passed': (passed == total),
        **watermark_result
    }
    analysis_results.append(result)

In [ ]:
results_df = pd.DataFrame(analysis_results)
results_df.to_csv(RESULTS_CSV, index=False)
print(f"Saved: {RESULTS_CSV}")

print(f"\nCode Quality Summary:")
print(f"Total: {len(results_df)}")
print(f"Avg pass rate: {results_df['pass_rate'].mean():.1f}%")
print(f"All tests passed: {results_df['all_passed'].sum()}")
print(f"Some failures: {(~results_df['all_passed']).sum()}")